# Lexicon: The Unwriting — 12 Villagers Character Forge Pipeline (Colab Pro+ GPU)

**100% Plajah-Native Tools (Zero Blender!)** offloading all heavy GPU + CPU work off the local computer onto **Colab Pro+ A100 / L4 GPU**:
1. **GPU Matting (`rembg` `isnet-general-use`)** + ground-shadow cleanup → crisp RGBA cutouts for all 12 villagers (`front`, true `right`/`left`, `back`)
2. **GPU Multi-View 3D Sculpt (`tencent/Hunyuan3D-2mv`)** using all 3 true-orientation views so asymmetric props (Mira's tome/pouch, Tolo's runic hammer, Thorne's staff, Oren's cane & scroll case, Lyra's lyre, Bram's shield, Maya's vials, Zora's map tube) align cleanly in 3D
3. **Plajah Silhouette Refinement (`fit_silhouette.py`)** + **Plajah 2048px Multi-View Art Projection (`paint_from_views.py`)** with true-side priority (no prop ghosting onto the wrong hip!)
4. **Plajah Sculpted High-Poly → Low-Poly Normal + 3D Bunnell AO + Lexicon Painted-Real Matte PBR Baker** (`village_cloth` / `village_leather` / `village_armor` + subtle emissive runes/herbs/vials for Tolo, Kaelen & Maya)
5. **Plajah Proportion-Aware Contract Humanoid Rig (`adult`/`elder`/`youth`/`child` for Pip!) + Screened Poisson Metric Bone-Heat Skinning** + **Multi-View Fidelity Scorer** (`front`, true `side`, `back` delta sheets)
6. Packages **`lexicon_cast_outputs.zip`** for fast local 58-joint Character Forge (`forgeRigMesh.ts`) ingest!

**Before running:** Set **Runtime → Change runtime type → `A100 GPU` (recommended) or `L4 GPU`**, and upload `lexicon_cast_inputs.zip` to `/content/` (or Google Drive).

In [ ]:
# Cell 1: Verify CUDA GPU and install pure wheel dependencies (Zero Blender, zero CUDA compile)
import os, subprocess, sys, torch

assert torch.cuda.is_available(), 'No CUDA GPU attached! Go to Runtime -> Change runtime type -> A100 GPU or L4 GPU.'
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f'GPU Ready: {gpu_name} ({vram_gb:.1f} GB VRAM), PyTorch {torch.__version__}')

if not os.path.isdir('/content/Hunyuan3D-2'):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Tencent/Hunyuan3D-2.git', '/content/Hunyuan3D-2'], check=True)

pkgs = [
    'trimesh', 'einops', 'omegaconf', 'safetensors', 'huggingface_hub',
    'scikit-image', 'pymeshlab', 'opencv-python-headless', 'scipy',
    'pygltflib', 'accelerate', 'diffusers', 'transformers', 'timm',
    'rembg', 'onnxruntime-gpu', 'xatlas', 'matplotlib', 'Pillow'
]
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])
if r.returncode != 0:
    # Fallback to CPU onnxruntime if onnxruntime-gpu wheel conflicts with Colab CUDA
    pkgs = [p if p != 'onnxruntime-gpu' else 'onnxruntime' for p in pkgs]
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], check=True)

if '/content/Hunyuan3D-2' not in sys.path:
    sys.path.insert(0, '/content/Hunyuan3D-2')

import hy3dgen, rembg, xatlas, pymeshlab, trimesh
print('All Plajah + Hunyuan3D-2mv dependencies OK!')

In [ ]:
# Cell 2: Locate & unpack lexicon_cast_inputs.zip (supports /content/ upload OR Google Drive)
import glob, importlib, json, os, shutil, sys, zipfile

USE_GOOGLE_DRIVE = False  # Set True to read/write zips directly on Google Drive (/content/drive/MyDrive/plajah_lexicon_cast/)
INPUT_ZIP = None          # Leave None to auto-detect *lexicon_cast*inputs*.zip

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/plajah_lexicon_cast'
    os.makedirs(DRIVE_DIR, exist_ok=True)
    if INPUT_ZIP is None:
        cands = sorted(glob.glob(os.path.join(DRIVE_DIR, '*lexicon*inputs*.zip')), key=os.path.getmtime, reverse=True)
        if cands:
            INPUT_ZIP = cands[0]

if INPUT_ZIP is None:
    cands = sorted(glob.glob('/content/*lexicon*inputs*.zip'), key=os.path.getmtime, reverse=True)
    if cands:
        INPUT_ZIP = cands[0]
    else:
        from google.colab import files
        print('Upload lexicon_cast_inputs.zip via the Files sidebar or prompt...')
        uploaded = files.upload()
        INPUT_ZIP = os.path.join('/content', next(iter(uploaded.keys())))

assert os.path.isfile(INPUT_ZIP), f'Missing input zip: {INPUT_ZIP}'
print(f'Using input archive: {INPUT_ZIP} ({os.path.getsize(INPUT_ZIP) / 1e6:.2f} MB)')

WORK_IN = '/content/lexicon_cast_in'
WORK_OUT = '/content/lexicon_cast_out'
if os.path.isdir(WORK_IN):
    shutil.rmtree(WORK_IN)
os.makedirs(WORK_IN, exist_ok=True)
os.makedirs(WORK_OUT, exist_ok=True)

with zipfile.ZipFile(INPUT_ZIP, 'r') as zf:
    zf.extractall(WORK_IN)

if WORK_IN not in sys.path:
    sys.path.insert(0, WORK_IN)

import plajah_lexicon_forge as plf
importlib.reload(plf)

with open(os.path.join(WORK_IN, 'manifest.json'), 'r', encoding='utf-8') as f:
    MANIFEST = json.load(f)

print(f"Loaded {MANIFEST['batch']} ({len(MANIFEST['characters'])} villagers) + Plajah Forge toolkit:")
for item in MANIFEST['characters']:
    print(f"  - {item['id']:8s} ({item['proportions']:5s}, {item['height_m']:.2f}m, {item['pbr_profile']:15s}) views={list(item['mv_views'].keys())}")

In [ ]:
# Cell 3: Stage 1 — Colab GPU rembg (isnet-general-use) + ground-shadow cleanup -> crisp RGBA cutouts
import os, time
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from rembg import new_session, remove as rembg_remove

rembg_model = MANIFEST.get('rembg_model', 'isnet-general-use')
if '_REMBG_SESS' not in globals() or globals().get('_REMBG_MODEL') != rembg_model:
    print(f'Initializing rembg ({rembg_model})...')
    _REMBG_SESS = new_session(rembg_model)
    _REMBG_MODEL = rembg_model

def prepare_villager_cutout(src_path: str, dst_path: str) -> Image.Image:
    im = Image.open(src_path).convert('RGB')
    cut = rembg_remove(im, session=_REMBG_SESS)
    arr = np.array(cut.convert('RGBA'))
    alpha = arr[:, :, 3].copy()
    rgb = arr[:, :, :3].astype(np.float32)

    # Drop faint pale ground-shadow smudges around boots (low saturation, very bright, near bottom)
    alpha[alpha < 24] = 0
    ys0, _ = np.where(alpha >= 40)
    if len(ys0):
        h_fig = max(1, int(ys0.max() - ys0.min()))
        foot_band = np.zeros_like(alpha, dtype=bool)
        foot_band[max(0, int(ys0.max() - 0.06 * h_fig)) :, :] = True
        lum = rgb.mean(axis=2)
        sat = (rgb.max(axis=2) - rgb.min(axis=2)) / np.maximum(rgb.max(axis=2), 1.0)
        ground_shadow = foot_band & (lum > 188) & (sat < 0.14) & (alpha < 245)
        alpha[ground_shadow] = 0

    # Keep main silhouette + valid small features (staff scale pans, tassels >= 0.25% main area)
    n_lab, labels, stats, _ = cv2.connectedComponentsWithStats((alpha >= 30).astype(np.uint8), connectivity=8)
    if n_lab > 1:
        max_area = int(stats[1:, cv2.CC_STAT_AREA].max())
        keep = np.zeros_like(alpha, dtype=bool)
        for lid in range(1, n_lab):
            if stats[lid, cv2.CC_STAT_AREA] >= max(60, int(max_area * 0.0025)):
                keep |= (labels == lid)
        alpha[~keep] = 0
    arr[:, :, 3] = alpha

    ys, xs = np.where(alpha >= 30)
    if len(xs):
        y0, y1 = int(ys.min()), int(ys.max()) + 1
        x0, x1 = int(xs.min()), int(xs.max()) + 1
        crop = arr[y0:y1, x0:x1]
        ch, cw = crop.shape[:2]
        pad = max(20, int(max(ch, cw) * 0.06))
        canvas = np.zeros((ch + 2 * pad, cw + 2 * pad, 4), dtype=np.uint8)
        canvas[pad:pad + ch, pad:pad + cw] = crop
        arr = canvas
    out_im = Image.fromarray(arr, mode='RGBA')
    out_im.save(dst_path)
    return out_im

cutouts_by_id = {}
t0 = time.time()
for item in MANIFEST['characters']:
    cid = item['id']
    cdir = os.path.join(WORK_OUT, cid, 'cutouts')
    os.makedirs(cdir, exist_ok=True)
    cviews = {}
    for vk, fn in item['mv_views'].items():
        dst = os.path.join(cdir, f'{vk}.png')
        if os.path.isfile(dst):
            cviews[vk] = Image.open(dst).convert('RGBA')
        else:
            cviews[vk] = prepare_villager_cutout(os.path.join(WORK_IN, cid, fn), dst)
    cutouts_by_id[cid] = cviews
print(f'Prepared crisp RGBA cutouts for {len(cutouts_by_id)} villagers in {time.time() - t0:.1f}s')

# Render cutouts QA contact sheet
n_c = len(MANIFEST['characters'])
fig, axes = plt.subplots(n_c, 3, figsize=(9, 2.8 * n_c), facecolor='#1b1f2a')
if n_c == 1:
    axes = [axes]
for row, item in enumerate(MANIFEST['characters']):
    cid = item['id']
    for col, (vk, cim) in enumerate(cutouts_by_id[cid].items()):
        ax = axes[row][col]
        ax.set_facecolor('#2b3242')
        ax.axis('off')
        ax.imshow(cim)
        ax.set_title(f'{cid}: {vk}', color='white', fontsize=9.5)
plt.tight_layout()
cut_prev = os.path.join(WORK_OUT, 'qa_01_cutouts.png')
plt.savefig(cut_prev, dpi=120, facecolor='#1b1f2a')
plt.show()
print('Saved:', cut_prev)

In [ ]:
# Cell 4: Stage 2 — Colab GPU Multi-View Hunyuan3D-2mv Sculpt (front + true right/left + back)
import gc, json, os, sys, time
import numpy as np
import torch
import trimesh
import matplotlib.pyplot as plt

if '/content/Hunyuan3D-2' not in sys.path:
    sys.path.insert(0, '/content/Hunyuan3D-2')

from hy3dgen.shapegen import (
    Hunyuan3DDiTFlowMatchingPipeline,
    FloaterRemover,
    DegenerateFaceRemover,
    FaceReducer,
)
from hy3dgen.shapegen.models.conditioner import SingleImageEncoder, DualImageEncoder

# Compatibility shim for transformers >= 5.0 (DINOv2 renamed attention & SwiGLU keys)
def _patch_dinov2_v5(cls):
    if getattr(cls, '_v5_patched', False): return
    orig = cls.load_state_dict
    def _load(self, state_dict, strict=True, assign=False):
        expected = set(self.state_dict().keys())
        if any('.q_proj.' in k for k in expected) and any('.attention.query.' in k for k in state_dict):
            remapped = {}
            for k, v in state_dict.items():
                if '.mlp.weights_in.' in k:
                    g, u = v.chunk(2, dim=0)
                    remapped[k.replace('.mlp.weights_in.', '.mlp.gate_proj.')] = g.contiguous()
                    remapped[k.replace('.mlp.weights_in.', '.mlp.up_proj.')] = u.contiguous()
                elif '.mlp.weights_out.' in k:
                    remapped[k.replace('.mlp.weights_out.', '.mlp.down_proj.')] = v
                elif '.attention.attention.query.' in k:
                    remapped[k.replace('.attention.attention.query.', '.attention.q_proj.')] = v
                elif '.attention.attention.key.' in k:
                    remapped[k.replace('.attention.attention.key.', '.attention.k_proj.')] = v
                elif '.attention.attention.value.' in k:
                    remapped[k.replace('.attention.attention.value.', '.attention.v_proj.')] = v
                elif '.attention.output.dense.' in k:
                    remapped[k.replace('.attention.output.dense.', '.attention.o_proj.')] = v
                else:
                    remapped[k] = v
            state_dict = remapped
        return orig(self, state_dict, strict=strict, assign=assign)
    cls.load_state_dict = _load
    cls._v5_patched = True

_patch_dinov2_v5(SingleImageEncoder)
_patch_dinov2_v5(DualImageEncoder)

vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
# Full 30-step hunyuan3d-dit-v2-mv on A100/L4 (~25-35s/char -> ~6 min total for all 12, maximum silhouette & fold fidelity!)
# Set USE_TURBO = True for 8-step FlashVDM (~10-12s/char -> ~2 min total).
USE_TURBO = (vram_gb < 18.0)
subfolder = 'hunyuan3d-dit-v2-mv-turbo' if USE_TURBO else 'hunyuan3d-dit-v2-mv'

if '_HY3D_PIPE' not in globals() or globals().get('_HY3D_SUBFOLDER') != subfolder:
    print(f'Loading tencent/Hunyuan3D-2mv ({subfolder}, fp16) on {vram_gb:.1f} GB VRAM...')
    t_load = time.time()
    torch.set_default_dtype(torch.float16)
    try:
        with torch.device('cuda'):
            _HY3D_PIPE = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
                'tencent/Hunyuan3D-2mv',
                subfolder=subfolder,
                variant='fp16',
                use_safetensors=True,
            )
            if USE_TURBO:
                _HY3D_PIPE.enable_flashvdm(mc_algo='mc')
            else:
                try:
                    from hy3dgen.shapegen.models.autoencoders.surface_extractors import MCSurfaceExtractor
                    _HY3D_PIPE.vae.surface_extractor = MCSurfaceExtractor()
                except Exception as e:
                    print('Note: using default surface_extractor:', e)
    finally:
        torch.set_default_dtype(torch.float32)
    _HY3D_SUBFOLDER = subfolder
    gc.collect(); torch.cuda.empty_cache()
    print(f'Model loaded in {time.time() - t_load:.1f}s')
else:
    print(f'Reusing cached {subfolder} pipeline in GPU memory!')

pipeline = _HY3D_PIPE
floater_remover = FloaterRemover()
degen_remover = DegenerateFaceRemover()
face_reducer = FaceReducer()

shape_report = []
previews = []
for item in MANIFEST['characters']:
    cid = item['id']
    c_out = os.path.join(WORK_OUT, cid)
    os.makedirs(c_out, exist_ok=True)
    raw_glb = os.path.join(c_out, 'shape_raw.glb')
    clean_glb = os.path.join(c_out, 'shape_clean.glb')

    if os.path.isfile(clean_glb):
        print(f'[{cid:8s}] Cached shape_clean.glb, skipping sculpt.')
        mesh = trimesh.load(clean_glb, force='mesh')
        previews.append((cid, item['side_view'], mesh))
        continue

    views = cutouts_by_id[cid]
    steps = 8 if USE_TURBO else int(item.get('steps', 30))
    octree = int(item.get('octree_resolution', 384))
    face_limit = int(item.get('face_limit', 200000))
    seed = int(item.get('seed', 42))

    print(f'[{cid:8s}] Sculpting multi-view {list(views.keys())}, steps={steps}, octree={octree}...', flush=True)
    t0 = time.time()
    call_kwargs = dict(
        image=views,
        num_inference_steps=steps,
        octree_resolution=octree,
        num_chunks=20000,
        generator=torch.manual_seed(seed),
        output_type='trimesh',
    )
    try:
        mesh = pipeline(**call_kwargs, mc_algo='mc')[0]
    except TypeError:
        mesh = pipeline(**call_kwargs)[0]
    mesh = floater_remover(mesh)
    mesh = degen_remover(mesh)
    mesh = face_reducer(mesh, max_facenum=face_limit)
    mesh.export(raw_glb)
    cl_info = plf.clean_shape(raw_glb, clean_glb, target_height=float(item['height_m']))
    dt = time.time() - t0
    print(f"[{cid:8s}] Done in {dt:.1f}s -> {cl_info['faces']:,} faces, extents={cl_info['extents']}", flush=True)
    shape_report.append({'id': cid, 'seconds': round(dt, 1), **cl_info})
    previews.append((cid, item['side_view'], trimesh.load(clean_glb, force='mesh')))
    gc.collect(); torch.cuda.empty_cache()

# Free Hunyuan3D VRAM before downstream CPU/GPU stages
gc.collect(); torch.cuda.empty_cache()

# Render 3-view orthographic point-cloud QA sheet
fig, axes = plt.subplots(len(previews), 3, figsize=(9, 2.8 * len(previews)), facecolor='#11131a')
if len(previews) == 1:
    axes = [axes]
for row, (cid, sk, mesh) in enumerate(previews):
    v = np.asarray(mesh.vertices)
    vs = v[::max(1, len(v) // 18000)]
    side_proj = ('right', vs[:, 2], vs[:, 1], -vs[:, 0]) if sk == 'right' else ('left', -vs[:, 2], vs[:, 1], vs[:, 0])
    for col, (lbl, px, py, depth) in enumerate([('front', vs[:, 0], vs[:, 1], vs[:, 2]), side_proj, ('back', -vs[:, 0], vs[:, 1], -vs[:, 2])]):
        ax = axes[row][col]
        ax.set_facecolor('#11131a')
        order = np.argsort(depth)
        ax.scatter(px[order], py[order], c=depth[order], cmap='viridis', s=0.6, linewidths=0)
        ax.set_aspect('equal'); ax.axis('off')
        ax.set_title(f'{cid} ({lbl})', color='white', fontsize=9.5)
plt.tight_layout()
prev_path = os.path.join(WORK_OUT, 'qa_02_shapes.png')
plt.savefig(prev_path, dpi=130, facecolor='#11131a')
plt.show()
print('Saved:', prev_path)

In [ ]:
# Cell 5: Stage 3 & 4 — Plajah Silhouette Fit + 2048px Multi-View Art Projection + Sculpted HP->LP PBR Bake
import os, subprocess, sys, time

pbr_report = []
FORCE_REPAINT = True  # Re-run tear-proof 3-view fit + crisp paint on cached shape_clean.glb (skips slow sculpt!)
for item in MANIFEST['characters']:
    cid = item['id']
    sk = item['side_view']
    opp = 'left' if sk == 'right' else 'right'
    c_out = os.path.join(WORK_OUT, cid)
    cut_dir = os.path.join(c_out, 'cutouts')
    pad_dir = os.path.join(c_out, 'padded_refs')
    os.makedirs(pad_dir, exist_ok=True)

    clean_glb = os.path.join(c_out, 'shape_clean.glb')
    fit_glb = os.path.join(c_out, 'shape_fit.glb')
    painted_glb = os.path.join(c_out, 'painted.glb')
    pbr_glb = os.path.join(c_out, 'pbr.glb')

    if (not FORCE_REPAINT) and os.path.isfile(pbr_glb):
        print(f'[{cid:8s}] Cached pbr.glb, skipping paint & PBR bake.')
        continue

    t0 = time.time()
    # 1. RGB-pad transparent fringes so bilinear art projection never bleeds seams
    plf.pad_transparent_rgb(os.path.join(cut_dir, 'front.png'), os.path.join(pad_dir, 'front.png'), flip_x=False)
    plf.pad_transparent_rgb(os.path.join(cut_dir, 'back.png'), os.path.join(pad_dir, 'back.png'), flip_x=False)
    has_side = os.path.isfile(os.path.join(cut_dir, f'{sk}.png'))
    if has_side:
        plf.pad_transparent_rgb(os.path.join(cut_dir, f'{sk}.png'), os.path.join(pad_dir, f'{sk}.png'), flip_x=False)
        plf.pad_transparent_rgb(os.path.join(cut_dir, f'{sk}.png'), os.path.join(pad_dir, f'{opp}.png'), flip_x=True)

    # 2. Tear-proof 3-view consensus silhouette refinement (front + side + back jointly, zero prop tearing)
    if has_side:
        finfo = plf.fit_silhouette_views(clean_glb, cut_dir, fit_glb, side_key=sk, iters=4, damp=0.48, mesh_smooth=24)
    else:
        import shutil as _sh; _sh.copy2(clean_glb, fit_glb)
        finfo = {'before_front': 0.0, 'after_front': 0.0, 'before_side': 0.0, 'after_side': 0.0, 'before_back': 0.0, 'after_back': 0.0}

    # 3. Plajah 2048px multi-view art projection + quadric decimation (56k faces) + xatlas UV
    cmd = [
        sys.executable, os.path.join(WORK_IN, 'paint_from_views.py'),
        '--mesh', fit_glb,
        '--out', painted_glb,
        '--faces', str(item.get('game_faces', 56000)),
        '--res', str(item.get('tex_res', 2048)),
        '--min-facing', '0.18',
        '--facing-pow', '6.0',
        '--view', f"front={os.path.join(pad_dir, 'front.png')}",
        '--view', f"back={os.path.join(pad_dir, 'back.png')}",
        '--view-weight', 'back=1.0',
    ]
    if has_side:
        cmd += ['--view', f"{sk}={os.path.join(pad_dir, f'{sk}.png')}", '--view', f"{opp}={os.path.join(pad_dir, f'{opp}.png')}", '--view-weight', f'{sk}=0.85', '--view-weight', f'{opp}=0.18']
    r = subprocess.run(cmd, cwd=WORK_IN, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-1200:], r.stderr[-1200:])
        raise RuntimeError(f'paint_from_views failed on {cid}')

    # 4. Plajah 200k HP -> 56k LP sculpted normal + 3D Bunnell AO + Lexicon painted-real PBR + optional emissive
    pinfo = plf.bake_lexicon_pbr(
        painted_glb,
        fit_glb,
        pbr_glb,
        profile=item.get('pbr_profile', 'village_cloth'),
        emissive=item.get('emissive'),
    )
    dt = time.time() - t0
    print(
        f"[{cid:8s}] {dt:4.1f}s | fit F:{finfo['before_front']:.2f}->{finfo['after_front']:.2f} "
        f"S:{finfo['before_side']:.2f}->{finfo['after_side']:.2f} B:{finfo['before_back']:.2f}->{finfo['after_back']:.2f} | "
        f"HP-nrm={'YES' if pinfo['hp_normal'] else 'no'} AO={pinfo['mean_ao']:.2f} R={pinfo['mean_rough']:.2f} "
        f"M={pinfo['mean_metal']:.2f} em={pinfo['emissive_pct']:.1f}% ({pinfo['mb']:.1f} MB)",
        flush=True,
    )
    pbr_report.append({'id': cid, 'fit': finfo, 'pbr': pinfo, 'sec': round(dt, 1)})

In [ ]:
# Cell 6: Stage 5 & 6 — Proportion-Aware Contract Rig + Screened Poisson Skin + Multi-View Fidelity Scoring
import json, os, time
import matplotlib.pyplot as plt
from PIL import Image

rig_scores = []
for item in MANIFEST['characters']:
    cid = item['id']
    sk = item['side_view']
    c_out = os.path.join(WORK_OUT, cid)
    pbr_glb = os.path.join(c_out, 'pbr.glb')
    rig_glb = os.path.join(c_out, 'rigged_22.glb')
    j_json = os.path.join(c_out, 'joints.json')
    sc_dir = os.path.join(c_out, 'score')

    t0 = time.time()
    if item.get('rig_mode', 'humanoid') == 'quadruped':
        import shutil as _sh; _sh.copy2(pbr_glb, rig_glb)
        rinfo = {'p99_stretch': 1.0, 'max_stretch': 1.0, 'over_3x': 0}
    else:
        rinfo = plf.build_contract_22_glb(
            pbr_glb,
            rig_glb,
            j_json,
            prop_profile=item['proportions'],
            has_robe=bool(item.get('has_robe', False)),
            has_left_pole=bool(item.get('has_left_pole', False)),
        )
    sinfo = plf.score_glb_views_software(rig_glb, os.path.join(c_out, 'cutouts'), sk, sc_dir)
    dt = time.time() - t0
    ov = sinfo['overall'] * 100.0
    gate = 'PASS >=95%' if ov >= 95.0 else f'{95.0 - ov:.1f}% short'
    s_str = f"{sinfo['side']['score']*100:4.1f}%" if ('side' in sinfo and not sinfo['side'].get('skipped')) else ' n/a '
    print(
        f"[{cid:8s}] {dt:4.1f}s | overall={ov:5.1f}% ({gate:11s}) | "
        f"F:{sinfo['front']['score']*100:4.1f}% S:{s_str} B:{sinfo['back']['score']*100:4.1f}% | "
        f"stretch p99.5={rinfo['p99_stretch']:.2f}x max={rinfo['max_stretch']:.2f}x",
        flush=True,
    )
    rig_scores.append({'id': cid, 'name': item['name'], 'rig': rinfo, 'scores': sinfo})

with open(os.path.join(WORK_OUT, 'scores_summary.json'), 'w', encoding='utf-8') as f:
    json.dump({'batch': MANIFEST['batch'], 'results': rig_scores}, f, indent=2)

# Render visual delta contact sheet (front | side | back) for all 12 villagers!
fig, axes = plt.subplots(len(rig_scores), 3, figsize=(15, 3.2 * len(rig_scores)), facecolor='#12141c')
if len(rig_scores) == 1:
    axes = [axes]
for row, entry in enumerate(rig_scores):
    cid = entry['id']
    sinfo = entry['scores']
    for col, lbl in enumerate(('front', 'side', 'back')):
        ax = axes[row][col]
        ax.axis('off')
        dp = os.path.join(WORK_OUT, cid, 'score', f'{lbl}_delta.png')
        if not os.path.isfile(dp) or lbl not in sinfo:
            ax.set_title(f'{cid} {lbl} (n/a)', color='#888', fontsize=9)
            continue
        ax.imshow(Image.open(dp))
        v = sinfo[lbl]
        ax.set_title(f"{cid} {lbl} ({v['score']*100:.1f}% | IoU {v['iou']:.2f} col {v['colour']:.2f})", color='white', fontsize=9)
plt.tight_layout()
qa_delta = os.path.join(WORK_OUT, 'qa_03_fidelity_deltas.png')
plt.savefig(qa_delta, dpi=110, facecolor='#12141c')
plt.show()
print('Saved:', qa_delta)

In [ ]:
# Cell 7: Package lexicon_cast_outputs.zip (rigged PBR GLBs + crisp cutouts + joints + QA sheets) -> Drive or download
import os, shutil, zipfile

INCLUDE_RAW_HP = False  # Keep False for lean ~60-80 MB download; set True to also bundle 200k shape_raw.glb
ZIP_OUT = '/content/lexicon_cast_outputs.zip'

with zipfile.ZipFile(ZIP_OUT, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for top_fn in ('scores_summary.json', 'qa_01_cutouts.png', 'qa_02_shapes.png', 'qa_03_fidelity_deltas.png'):
        fp = os.path.join(WORK_OUT, top_fn)
        if os.path.isfile(fp):
            zf.write(fp, top_fn)
    zf.write(os.path.join(WORK_IN, 'manifest.json'), 'manifest.json')
    for item in MANIFEST['characters']:
        cid = item['id']
        c_out = os.path.join(WORK_OUT, cid)
        keep_files = [
            ('rigged_22.glb', f'{cid}/rigged_22.glb'),
            ('joints.json', f'{cid}/joints.json'),
            ('score/scores.json', f'{cid}/score/scores.json'),
            ('score/front_delta.png', f'{cid}/score/front_delta.png'),
            ('score/side_delta.png', f'{cid}/score/side_delta.png'),
            ('score/back_delta.png', f'{cid}/score/back_delta.png'),
        ]
        if INCLUDE_RAW_HP:
            keep_files.append(('shape_raw.glb', f'{cid}/shape_raw.glb'))
        for rel_src, arc in keep_files:
            fp = os.path.join(c_out, rel_src)
            if os.path.isfile(fp):
                zf.write(fp, arc)
        cut_dir = os.path.join(c_out, 'cutouts')
        if os.path.isdir(cut_dir):
            for fn in sorted(os.listdir(cut_dir)):
                if fn.endswith('.png'):
                    zf.write(os.path.join(cut_dir, fn), f'{cid}/cutouts/{fn}')

mb = os.path.getsize(ZIP_OUT) / 1e6
print(f'READY: {ZIP_OUT} ({mb:.2f} MB) for {len(MANIFEST["characters"])} Lexicon villagers!')

if USE_GOOGLE_DRIVE:
    dst = os.path.join(DRIVE_DIR, 'lexicon_cast_outputs.zip')
    shutil.copyfile(ZIP_OUT, dst)
    print('Copied to Google Drive:', dst)
else:
    from google.colab import files
    files.download(ZIP_OUT)